# 00 — Get Data

## Student Notebook

This notebook downloads all raw data files required for this project into the project root directory. Run it **once** before opening the exploratory notebooks.

### Project layout expected after running this notebook

```
UW-SysBiol-Project2/
├── single-cell-tracks_exp1-6_noErbB2.csv.gz   ← main data  (~625 MB, downloaded here)
├── 01-readme-experiment-description_2022-04-05.csv          ← metadata
├── requirements.txt                                          ← Python dependencies
├── setup_env.sh                                              ← environment setup script
├── students-notebooks/
└── teacher-notebooks/
```

The goal here is practical and simple: prepare the files, verify that they exist, and then move on to the analysis notebooks.

## 1 — Environment Setup

Before running this notebook, create a dedicated virtual environment with all required packages.
Open a terminal in the **project root** and run the single line below:

```bash
bash setup_env.sh
```

Or copy-paste the full one-liner directly:

```bash
python3 -m venv .venv && source .venv/bin/activate && pip install -U pip && pip install -r requirements.txt && python -m ipykernel install --user --name=sysbiol-p2 --display-name="Python (SysBiol Project 2)" && echo "✓ Environment ready. Select kernel 'Python (SysBiol Project 2)' in VS Code / Jupyter."
```

After that, select the **"Python (SysBiol Project 2)"** kernel in VS Code (bottom-right kernel picker) or Jupyter before running any cell.


## 2 — Install `gdown` (if needed)

`gdown` is a small utility that handles Google Drive's virus-scan confirmation page,
which breaks plain `curl`/`wget` for files larger than ~100 MB.
The cell below installs it quietly if it is not already present in the current kernel.


In [ ]:
import importlib, subprocess, sys

def _ensure(pkg, install_name=None):
    """Import *pkg*; install via pip if missing."""
    try:
        importlib.import_module(pkg)
        print(f"✓ {pkg} already installed")
    except ModuleNotFoundError:
        print(f"  Installing {install_name or pkg} …")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", install_name or pkg])
        print(f"✓ {pkg} installed")

_ensure("gdown")
_ensure("tqdm")


## 3 — Define Paths and File Catalogue

All paths are resolved relative to the project root (one level above `notebooks/`).
Add additional Google Drive files to `DRIVE_FILES` if needed.


In [ ]:
from pathlib import Path

# ── Project root (one level above this notebook) ────────────────────────────
ROOT = Path(__file__).resolve().parent.parent if "__file__" in dir() else Path.cwd().parent
if not (ROOT / "requirements.txt").exists():
    ROOT = Path.cwd()          # fallback: notebook run from root itself
ROOT = ROOT.resolve()
print("Project root:", ROOT)

# ── Catalogue of files to download from Google Drive ────────────────────────
# Each entry: (filename, Google-Drive-file-id, expected_size_mb)
DRIVE_FILES = [
    (
        "single-cell-tracks_exp1-6_noErbB2.csv.gz",
        "1nViMkO1WqJOEF9rvkOtfwyab3-YtCWVu",
        625,
    ),
]

# ── Files that should already be present (sanity check) ─────────────────────
LOCAL_FILES = [
    "01-readme-experiment-description_2022-04-05.csv",
]


## 4 — Download from Google Drive

The cell below skips any file that is already present and has a non-zero size.
Re-run to resume an interrupted download.


In [ ]:
import gdown

for filename, file_id, expected_mb in DRIVE_FILES:
    dest = ROOT / filename

    if dest.exists() and dest.stat().st_size > 0:
        actual_mb = dest.stat().st_size / 1024**2
        print(f"✓ {filename} already present ({actual_mb:.0f} MB) — skipping download.")
        continue

    print(f"⬇  Downloading {filename}  (~{expected_mb} MB) …")
    url = f"https://drive.google.com/uc?id={file_id}"
    gdown.download(url, output=str(dest), quiet=False, fuzzy=True)
    actual_mb = dest.stat().st_size / 1024**2
    print(f"✓ {filename} saved ({actual_mb:.0f} MB)")


## 5 — Verify All Files


In [ ]:
import pandas as pd

all_expected = [(f, None) for f in LOCAL_FILES] + [(f, fid) for f, fid, _ in DRIVE_FILES]

rows = []
all_ok = True
for filename, _ in all_expected:
    p = ROOT / filename
    exists = p.exists()
    size_mb = p.stat().st_size / 1024**2 if exists else None
    status = "✓" if exists else "✗ MISSING"
    if not exists:
        all_ok = False
    rows.append({"file": filename, "status": status, "size_MB": f"{size_mb:.1f}" if size_mb else "—"})

summary = pd.DataFrame(rows)
print(summary.to_string(index=False))

if all_ok:
    print("\n✓ All files present. You are ready to open notebook 01.")
else:
    print("\n✗ Some files are missing. Re-run section 4.")


## 6 — Quick Preview

Read the first 3 rows of the main data file to confirm the gzip archive is intact and the columns look correct.


In [ ]:
data_path = ROOT / "single-cell-tracks_exp1-6_noErbB2.csv.gz"
preview = pd.read_csv(data_path, compression="gzip", nrows=3)

print(f"Columns ({len(preview.columns)}):")
print(list(preview.columns))
print()
preview
